# 2 · Fine-tuning con historias clínicas sintéticas

**Inputs:** `codigos-test` y el Output del notebook generador (`synth.zip`) · **Sesión:** GPU T4 x2, Internet On

Split por historia: **60% train / 20% val / 20% test**.

Antes de Save & Run All: corre a mano C1 y pon la ruta del zip en C2.

## Parte 0 · Instalación y scripts

In [ ]:
# 0. Verificar INTERNET (si falla: panel derecho -> Session options -> Internet ON, y vuelve a guardar la versión)
import socket
for host in ("pypi.org", "huggingface.co"):
    try:
        socket.create_connection((host, 443), timeout=10).close()
        print(f"OK  {host}")
    except OSError as e:
        raise SystemExit(f"SIN INTERNET: no llego a {host} ({e}). Activa Internet en Session options.")

In [ ]:
!pip install -q "transformers>=4.49" peft bitsandbytes accelerate qwen-vl-utils editdistance

In [ ]:
# Verifica que los scripts estén en el input
!ls /kaggle/input/datasets/andyjohan/codigos-test/

In [ ]:
# Copia los scripts a working
CODE = "/kaggle/input/datasets/andyjohan/codigos-test"
!cp {CODE}/*.py /kaggle/working/
%cd /kaggle/working
!ls *.py

In [ ]:
# Verifica que sean las versiones correctas
!grep -c "warmup_steps=warmup_steps" train_qwen_lora.py   # debe dar 1
!grep -c "fp16" eval_cer.py                               # debe dar 2

## Parte C · Historias clínicas sintéticas

Antes: **Add Input → Notebooks → Your Work →** el notebook que generó `synth.zip`.

In [ ]:
# C1. Ubicar synth.zip
%cd /kaggle/working
!find /kaggle/input -name "synth.zip"

In [ ]:
# C2. Descomprimir (CAMBIA la ruta por la que mostró C1)
ZIP = "/kaggle/input/notebooks/andyjohan/data-sintetica/synth.zip"
!unzip -q -o {ZIP} -d /kaggle/working
!ls synth
!wc -l synth/lines.jsonl

In [ ]:
# C3. Separar train/val/test POR HISTORIA (60/20/20)
import json, random
from pathlib import Path

SRC = "/kaggle/working/synth"
OUT = Path("/kaggle/working/data_synth"); OUT.mkdir(exist_ok=True)

rows = [json.loads(l) for l in open(f"{SRC}/lines.jsonl", encoding="utf-8")]
for r in rows:
    r["image"] = f"{SRC}/lines/{Path(r['image']).name}"

pages = sorted({r["id"].rsplit("_", 1)[0] for r in rows})
random.Random(42).shuffle(pages)
n = len(pages)
grupo = {p: "train" for p in pages[:int(.6 * n)]}
grupo.update({p: "val" for p in pages[int(.6 * n):int(.8 * n)]})
grupo.update({p: "test" for p in pages[int(.8 * n):]})

for split in ("train", "val", "test"):
    sel = [r for r in rows if grupo[r["id"].rsplit("_", 1)[0]] == split]
    with open(OUT / f"{split}.jsonl", "w", encoding="utf-8") as f:
        for r in sel:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{split}: {len(sel)} líneas ({sum(1 for p in grupo.values() if p == split)} historias)")

assert Path(rows[0]["image"]).exists(), "Ruta mal: revisa SRC"
print("OK, ejemplo:", rows[0]["text"])

In [ ]:
# C4. Baseline zero-shot en test sintético (sin entrenar)
!python eval_cer.py --test data_synth/test.jsonl --qlora --fp16 --bs 4 \
   --max_pixels 401408 --model Qwen/Qwen2.5-VL-3B-Instruct --out_csv preds_synth_baseline.csv

In [ ]:
# C5. Entrenamiento con data sintética (~3 h con ~600 historias de train)
!python train_qwen_lora.py --train data_synth/train.jsonl --val data_synth/val.jsonl \
   --out runs/synth-lora --qlora --fp16 --bs 1 --grad_accum 16 --max_pixels 401408 \
   --epochs 2 --model Qwen/Qwen2.5-VL-3B-Instruct

In [ ]:
# C6. Examen en test sintético (después de entrenar)
!python eval_cer.py --test data_synth/test.jsonl --qlora --fp16 --bs 4 \
   --max_pixels 401408 --model Qwen/Qwen2.5-VL-3B-Instruct --adapter runs/synth-lora \
   --out_csv preds_synth_ft.csv

## Resultados · Tabla resumen

In [ ]:
import pandas as pd, editdistance, os

def resumen(csv, nombre):
    df = pd.read_csv(csv).fillna("")
    c_err = sum(editdistance.eval(r, h) for r, h in zip(df["ref"], df["hyp"]))
    c_tot = df["ref"].str.len().clip(lower=1).sum()
    w_err = sum(editdistance.eval(r.split(), h.split()) for r, h in zip(df["ref"], df["hyp"]))
    w_tot = sum(max(len(r.split()), 1) for r in df["ref"])
    return {"Experimento": nombre, "CER (%)": round(100 * c_err / c_tot, 2),
            "WER (%)": round(100 * w_err / w_tot, 2), "n": len(df)}

exps = [("preds_synth_baseline.csv", "Qwen2.5-VL-3B · zero-shot"),
        ("preds_synth_ft.csv", "Qwen2.5-VL-3B · LoRA sintético")]
pd.DataFrame([resumen(f, n) for f, n in exps if os.path.exists(f)])

## Guardar el modelo (limpieza del Output)

In [ ]:
# Deja en el Output solo lo importante: el modelo (zip) y los CSV de resultados
%cd /kaggle/working
!rm -rf synth data_synth                       # ~13,000 archivos que ya no se necesitan (siguen en synth.zip del generador)
!rm -rf runs/synth-lora/checkpoint-*           # checkpoints intermedios (pesados; el mejor ya está en runs/synth-lora)
import os
if os.path.exists("runs/synth-lora/adapter_model.safetensors"):
    !zip -qr synth-lora.zip runs/synth-lora    # el adaptador LoRA en un solo archivo
    !rm -rf runs
else:
    print("No se encontró el modelo entrenado: revisa la celda C5")
!ls -lh /kaggle/working